# 06 - Hybrid Evaluation

Combine ML scores, LLM sentiment, and MCDA ranking for end-to-end hybrid evaluation.

## Objectives
- Run full pipeline: Data -> Feature Eng -> ML -> (LLM) -> MCDA
- Compare hybrid vs ML-only vs LLM-only (when available)
- Evaluate using ModelEvaluator metrics

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.data import DataLoader, FeatureEngineer
from src.models.ml import MLTrainer, MLPredictor, ModelEvaluator
from src.mcda import ProjectRanker

## 1. Load and Prepare Data

In [ ]:
loader = DataLoader()
df = loader.load(Path('../data/raw/sample_projects.csv'))
fe = FeatureEngineer()
df = fe.create_features(df)

exclude = ['project_id', 'project_name', 'risk_level', 'status_comments', 'project_description', 
           'team_feedback', 'start_date', 'planned_end_date', 'actual_end_date', 'technology_stack', 'stakeholder_notes']
feature_cols = [c for c in df.columns if c not in exclude and df[c].dtype in ['int64', 'float64', 'int32', 'float32']]

X = df[feature_cols].fillna(0)
y = (df['risk_level'] == 'High').astype(int) if 'risk_level' in df.columns else (df['completion_rate'] < 50).astype(int)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. ML Training and Evaluation

In [ ]:
trainer = MLTrainer(model_type='random_forest')
trainer.train(X_train, y_train)

y_pred = trainer.model.predict(X_test)
y_proba = trainer.model.predict_proba(X_test)[:, -1] if hasattr(trainer.model, 'predict_proba') else y_pred

evaluator = ModelEvaluator()
eval_results = evaluator.evaluate(y_test, y_pred, y_proba)
evaluator.print_report()

## 3. MCDA Ranking with ML Scores

In [ ]:
df['risk_score'] = trainer.model.predict_proba(X)[:, -1]
if 'project_id' not in df.columns:
    df['project_id'] = df['project_name'] if 'project_name' in df.columns else range(len(df))

ranker = ProjectRanker()
rankings = ranker.rank(df)

print('Top 5 High-Risk (by MCDA):')
print(rankings.nsmallest(5, 'mcda_score')[['project_name', 'mcda_score', 'rank', 'risk_level']])

## 4. Compare ML-only vs Hybrid

In [ ]:
# ML-only: risk_score from model
ml_top5 = df.nsmallest(5, 'risk_score')[['project_name', 'risk_score']]

# Hybrid: MCDA combines ML + SPI + CPI + team_stability
mcda_top5 = rankings.nsmallest(5, 'mcda_score')[['project_name', 'mcda_score']]

print('ML-only top 5 risk:')
print(ml_top5)
print('\nMCDA hybrid top 5 risk:')
print(mcda_top5)